# Vision-Language Models (VLM): Make an LLM Understand Images

> **Background**: GPT-4V, Claude, and Gemini can all "talk about images". How do they do it?
>
> Goal for this part: **understand the core VLM idea: how to turn an image into a token sequence that an LLM can consume.**

One-sentence core idea:
**image -> split into patches -> each patch becomes a vector -> treat them as special "visual tokens" -> concatenate with text tokens -> feed to the LLM.**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

torch.manual_seed(42)

### 1. The core problem: images and text are totally different

```
Text: "a cat sits on a mat"
  -> Tokenizer -> [12, 45, 78, 3, 90, 23]   (1D token sequence)

Image: a 224x224 cat photo
  -> ??? -> [?, ?, ?, ...]                  (how to make tokens?)
```

LLMs only know token sequences. So the VLM task is:

**convert an image into a sequence of tokens, so the LLM can "read" it like text.**

A common 3-step recipe:
1. split the image into patches
2. turn each patch into a vector (patch embedding)
3. treat those vectors as tokens and concatenate with text tokens


### 2. Step 1: patchify the image

Like turning a big photo into a jigsaw puzzle. Each patch is a "visual word".

```
Original image: 224 x 224
Patch size: 16 x 16
-> 14 x 14 = 196 patches

+--+--+--+--+
|  |  |  |  |  each patch is 16x16 pixels
+--+--+--+--+
|  |🐱|  |  |  cat face is in one patch
+--+--+--+--+
|  |  |  |  |
+--+--+--+--+
```

Why 16x16? That was the choice in the original ViT paper, and it became a standard.


In [ ]:
#  image：3 (RGB) × 224 × 224
IMG_SIZE = 224
PATCH_SIZE = 16

#  image( )
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print(f" image : {fake_image.shape}")
print(f" : [3 , 224, 224]")

# split intopatch
# unfold ： 
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE).unfold(2, PATCH_SIZE, PATCH_SIZE)
print(f"\n : {patches.shape}")
print(f" : [3 , 14 patch, 14 patch, 16 , 16 ]")

#  [num_patches, patch_dim]
num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE  # 3×16×16 = 768
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print(f"\n : {patches_flat.shape}")
print(f" : [{num_patches} patch, patch {patch_dim} ]")
print(f"\n : text token 1 -> vision token {patch_dim} ")
print(f" ： {patch_dim} text token ")

### 3. Step 2: patch embedding (turn each patch into a vector)

Each patch has `3*16*16 = 768` raw pixel values. But the LLM embedding dimension might be 512/768/...

Use a linear projection from 768 -> d_model so visual tokens and text tokens live in the same vector space.

```
patch (768) -> Linear(768, d_model) -> visual token (d_model)
text  "cat"  -> embedding lookup     -> text token   (d_model)

Same dimension -> can be concatenated.
```


In [ ]:
class PatchEmbedding(nn.Module):
    """ imagesplit intopatch, patch vector ViT 「Tokenizer」—— image """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        #  ： , 「 + 」
        # kernel_size=patch_size, stride=patch_size -> split intopatch
        self.proj = nn.Conv2d(in_channels, d_model, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        """ : x shape = [batch, 3, 224, 224] : shape = [batch, num_patches, d_model] """
        x = self.proj(x)          # [batch, d_model, 14, 14]
        x = x.flatten(2)           # [batch, d_model, 196]
        x = x.transpose(1, 2)      # [batch, 196, d_model]
        return x

#  
patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)  # batch=2 image
visual_tokens = patch_emb(dummy_img)

print(f" image: {dummy_img.shape}")
print(f"vision token: {visual_tokens.shape}")
print(f" : [2 image, 196 visiontoken, 512 ]")
print(f"\n vision token text token ！ .")

### 4. Step 3: visual tokens + text tokens -> feed the LLM together

This is the key operation: concatenate both modalities into one token sequence.

```
Input sequence:
[<img>] [v1] [v2] ... [v196] [</img>] [please] [describe] [this] [image]
  ^       ^    ^           ^     ^        ^       ^        ^      ^
 special  patch tokens                special    text tokens
```

The LLM just sees a long vector sequence. It does not intrinsically care which came from image vs text.


In [ ]:
#  VLM concatenate
d_model = 512
text_vocab_size = 1000

#  text tokenizer ID
text_ids = torch.tensor([[5, 12, 78, 3, 90]])  # " "
text_emb = nn.Embedding(text_vocab_size, d_model)
text_vecs = text_emb(text_ids)  # [1, 5, 512]

# vision token( PatchEmbedding )
visual_vecs = torch.randn(1, 196, d_model)  #  196 vision token

#  ！
combined = torch.cat([visual_vecs, text_vecs], dim=1)

print(f"vision token: {visual_vecs.shape}")
print(f"text token: {text_vecs.shape}")
print(f"concatenate : {combined.shape}")
print(f"\n : [1 , 196+5=201 token, 512 ]")
print(f"LLM 201 token , 196 image")

### 5. A complete VLM architecture sketch

```
                 Image (224x224x3)
                      |
                      v
           +----------------------+   196 patches -> vectors
           |   Patch Embedding    |   (often Conv2d with kernel=16,stride=16)
           +----------+-----------+
                      | [196, d_model]
                      v
           +----------------------+   optional: a few Transformer layers
           |   Vision Encoder     |   to let visual tokens interact
           +----------+-----------+
                      |
      +---------------+---------------------------+
      |                                           |
      v                                           v
 [visual tokens 1..196]                    [text tokens ...]
      |                                           |
      +------------------ concatenate ------------+
                      |
                      v
              +---------------+
              |  LLM Backbone |
              | (Transformer) |
              +-------+-------+
                      |
                      v
            "This is a cat on a mat"
```

Key insight: **you often do not need to change the LLM itself**.
Just add an "image -> tokens" front-end.


### 6. Three common architectures

1. **Visual-token concatenation** (LLaVA-style)
2. **Cross-attention** (Flamingo-style)
3. **Q-Former compression** (BLIP-2-style)

They differ in where and how visual information is injected.


### 6.1 Visual tokens (LLaVA-style)

- simplest: treat image tokens like extra text tokens
- easiest to implement and reuse an existing LLM
- downside: images consume many tokens (196+), making inference slower


### 6.2 Cross-attention (Flamingo-style)

Text tokens attend to image features via cross-attention.

- advantage: image features do not necessarily consume long sequence length
- downside: you modify the LLM architecture (add cross-attn blocks)


### 6.3 Q-Former (BLIP-2-style)

Q-Former uses learnable queries to compress many visual tokens into fewer tokens.

Example: 196 visual features -> 32 query tokens.
This can speed up attention a lot.


### 7. Engineering details that matter

Common knobs:
- projector design: Linear -> MLP -> Q-Former
- special tokens: `<img>...</img>` boundaries
- positional encoding: 1D order vs 2D positions vs learned
- multi-resolution: resize vs AnyRes tiling
- which vision layer to use: often second-to-last layer preserves more low-level detail


### 7.1 Why use the penultimate vision layer?

Using the final vision layer can be too abstract and lose details needed for fine-grained questions.
The penultimate layer often keeps more useful low-level signals.


### 8. Training: usually two stages

VLMs are typically trained in two stages:

Stage 1: alignment / pretraining
- goal: align visual tokens with text space
- data: lots of image-caption pairs
- often train only the projector, freeze vision encoder + LLM

Stage 2: instruction tuning
- goal: answer questions about images
- data: image + question/answer pairs
- train projector + (some of) LLM; often keep vision encoder frozen

```
Stage 1: image -> projector -> frozen LLM -> caption
Stage 2: image + question -> LLM -> answer
```


### 9. A minimal PyTorch VLM

Now we implement a tiny VLM that runs.


---

## VLM Checklist

1. [ok] Core idea: image -> patches -> vectors -> visual tokens -> concatenate with text tokens
2. [ok] Patch embedding is the image "tokenizer" (Conv2d can do patchify+project)
3. [ok] After dimension alignment, concatenate visual and text tokens; the LLM can stay unchanged
4. [ok] Cross-attention: Q from text, K/V from image features; gating can protect base model behavior
5. [ok] Projector choices: Linear vs MLP vs Q-Former
6. [ok] Common freezing: vision encoder frozen, projector trained, LLM trained mainly in stage 2
7. [ok] Architecture tradeoff: Visual tokens vs Cross-attn vs Q-Former
8. [ok] Images can be 196+ tokens, often a main reason VLM inference is slower

One-sentence summary: give the LLM "eyes". The vision encoder sees, the projector translates, the LLM reasons and speaks.


In [ ]:
class MiniVLM(nn.Module):
    """ Visual Token VLM：PatchEmbedding + text embedding + TransformerEncoder"""
    def __init__(self, text_vocab_size=100, d_model=128, num_heads=2, num_layers=2):
        super().__init__()
        self.vision = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, d_model=d_model)
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, text_vocab_size)

    def forward(self, images, text_ids):
        visual_tokens = self.vision(images)
        text_tokens = self.text_embedding(text_ids)
        x = torch.cat([visual_tokens, text_tokens], dim=1)
        x = self.layers(x)
        x = self.norm(x)
        return self.lm_head(x)

#  MiniVLM
vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

#  
dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))  # 10 text token

logits = vlm(dummy_images, dummy_text)

print(f" : image [{dummy_images.shape}] + text [{dummy_text.shape}]")
print(f" : {logits.shape}")
print(f" : [1 , 196+10=206 , 100 ]")
print(f"\n : {sum(p.numel() for p in vlm.parameters()):,}")

---

## One-line Summary

VLMs work by turning images into token-like vectors and feeding them into an LLM along with text.
